# dist-send-recv-pair — ex2: ring-pass via send/recv — every rank collects from all others

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dist-send-recv-pair`. Running the final beacon cell reports progress against the `Distributed: dist.send/recv pair` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: dist.send/recv pair` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dist-send-recv-pair`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dist-send-recv-pair"
DD_SUBTOPIC = "Distributed: dist.send/recv pair"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Ring-pass via send/recv — N-1 rotations around the ring

Ex1 used the linear `src loops sends to every other` pattern — fine for broadcast-from-one. The **ring-pass** is a different topology, where every rank pushes data to its right neighbor and receives from its left, repeated `world_size - 1` times. After all rotations, rank `r` has accumulated `world_size - 1` payloads from other ranks.

```python
left  = (rank - 1) % world_size
right = (rank + 1) % world_size
buf = my_payload.clone()
received = []
for _ in range(world_size - 1):
    dist.send(buf, dst=right)
    in_buf = t.zeros_like(buf)
    dist.recv(in_buf, src=left)
    received.append(in_buf.clone())
    buf = in_buf
```

**Why ring topology.** It's the structural backbone of `nccl`'s all-reduce — bandwidth scales O(1) per rank (each link carries the same volume regardless of `world_size`). Implementing it by hand once burns the pattern in.

**Deadlock trap.** `dist.send` is BLOCKING on gloo. If every rank calls `send` first and `recv` second, you must trust that the underlying transport buffers small messages — which gloo does for tensors below a few KB. For large tensors, use `dist.isend`/`irecv` and explicit `wait()`. This drill stays in the small-message regime.

### Exercise 2 — ring-pass via send/recv — every rank collects from all others

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply paired `dist.send(buf, dst=right)` + `dist.recv(in_buf, src=left)` in a loop of `world_size - 1` rotations to collect every other rank's payload at every rank — the ring-allreduce topology in its barest form.
> Keywords: send, recv, ring, rotation, topology
> ```

**KCs targeted:** `ring-neighbor-left-right-modular`, `n-minus-1-rotations`

Implement `ex2_ring_collect(rank, world_size, dist_module, my_payload)`. The ring-pass collector:

1. Compute neighbors: `left = (rank - 1) % world_size`, `right = (rank + 1) % world_size`.
2. Initialize `buf = my_payload.clone()` (the payload moving around the ring) and `received = []` (the list of payloads this rank has seen).
3. Loop `world_size - 1` times:
   a. `dist_module.send(buf, dst=right)` — push to right neighbor.
   b. Allocate fresh `in_buf = t.zeros_like(buf)`.
   c. `dist_module.recv(in_buf, src=left)` — receive from left.
   d. Append `in_buf.clone()` to `received`.
   e. Set `buf = in_buf` so the next rotation passes the just-received payload onward.
4. Return `received` — a list of `world_size - 1` tensors, in the order they were received.

After `world_size - 1` rotations, every rank has seen every OTHER rank's original payload (in some order; the order depends on rank, which we don't assert).

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `my_payload` — a 1-D float tensor unique to this rank.
Output: `list[Tensor]` of length `world_size - 1`.

In [ ]:
def ex2_ring_collect(rank: int, world_size: int, dist_module, my_payload: Tensor) -> list:
    """Ring-pass collector: every rank ends with all OTHER ranks' payloads."""
    raise NotImplementedError()


def _test_ex2():

    import threading
    import types as _types
    import torch as _t_for_fake

    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'

    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            self.scratch = {}
            self.tls = threading.local()
            self.results = [None] * world_size
            # send/recv mailbox keyed by (src, dst)
            self.mailbox = {}
            self.mailbox_cv = threading.Condition(self.lock)
        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['ar']
            if op == 'SUM':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced + x
            elif op == 'MAX':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.maximum(reduced, x)
            elif op == 'MIN':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.minimum(reduced, x)
            elif op == 'PROD':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced * x
            else:
                raise ValueError(f'unknown fake op {op!r}')
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()
        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['rd']
                if op == 'SUM':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced + x
                elif op == 'MAX':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.maximum(reduced, x)
                elif op == 'MIN':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.minimum(reduced, x)
                elif op == 'PROD':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced * x
                else:
                    raise ValueError(f'unknown fake op {op!r}')
                tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()
        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()
        def barrier_op(self):
            self.barrier.wait()
        def send(self, tensor, dst):
            rank = self.tls.rank
            with self.mailbox_cv:
                self.mailbox.setdefault((rank, dst), []).append(tensor.detach().clone())
                self.mailbox_cv.notify_all()
        def recv(self, tensor, src):
            rank = self.tls.rank
            with self.mailbox_cv:
                while not self.mailbox.get((src, rank)):
                    self.mailbox_cv.wait(timeout=10)
                payload = self.mailbox[(src, rank)].pop(0)
            tensor.copy_(payload)

    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size
        def _runner(rank):
            world.tls.rank = rank
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.send = lambda tensor, dst: world.send(tensor, dst)
            fake_dist.recv = lambda tensor, src: world.recv(tensor, src)
            fake_dist.init_process_group = lambda **kw: world.scratch.setdefault('_init_calls', []).append(kw)
            fake_dist.destroy_process_group = lambda: world.scratch.setdefault('_destroy_calls', []).append(rank)
            try:
                worker_fn(rank, world_size, fake_dist, world)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())
        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world


    # Each rank's payload is a unique scalar tensor.
    def _worker(rank, world_size, dist_module, world):
        my_payload = t.tensor([float(rank + 1) * 10.0])
        received = ex2_ring_collect(rank, world_size, dist_module, my_payload)
        # Convert to a sorted list of scalars for the test.
        world.results[rank] = sorted(r.item() for r in received)

    w = _run_fake_world(_worker, 4)
    # 4 ranks → each rank should end with 3 received payloads.
    # Payloads sent: [10, 20, 30, 40]. Each rank receives the OTHER three.
    for rank in range(4):
        got = w.results[rank]
        assert got is not None, f'rank {rank} returned None'
        assert len(got) == 3, f'rank {rank}: expected 3 received, got {len(got)}: {got}'
        expected = sorted([10.0, 20.0, 30.0, 40.0]) 
        expected.remove((rank + 1) * 10.0)   # own payload excluded
        for g, e in zip(got, expected):
            assert abs(g - e) < 1e-5, f'rank {rank}: got {got}, expected {expected}'

    # 2-rank case — only 1 rotation, each rank receives the other's payload.
    def _worker2(rank, world_size, dist_module, world):
        my_payload = t.tensor([float(rank + 1) * 100.0])
        received = ex2_ring_collect(rank, world_size, dist_module, my_payload)
        world.results[rank] = [r.item() for r in received]

    w2 = _run_fake_world(_worker2, 2)
    assert w2.results[0] == [200.0], f'2-rank rank 0 expected [200], got {w2.results[0]}'
    assert w2.results[1] == [100.0], f'2-rank rank 1 expected [100], got {w2.results[1]}'

    # Single-rank degenerate — 0 rotations, empty received list.
    def _worker1(rank, world_size, dist_module, world):
        my_payload = t.tensor([42.0])
        received = ex2_ring_collect(rank, world_size, dist_module, my_payload)
        world.results[rank] = received

    w1 = _run_fake_world(_worker1, 1)
    assert w1.results[0] == [], f'1-rank case: expected empty list, got {w1.results[0]}'

    # Vector payload (not scalar) — ring also works.
    def _worker_vec(rank, world_size, dist_module, world):
        my_payload = t.tensor([float(rank), float(rank) + 0.5, float(rank) + 0.25])
        received = ex2_ring_collect(rank, world_size, dist_module, my_payload)
        world.results[rank] = sorted(r.sum().item() for r in received)

    w_vec = _run_fake_world(_worker_vec, 3)
    # Each rank's payload sum: rank r → 3r + 0.75. Each rank receives the other two sums.
    for rank in range(3):
        own_sum = 3.0 * rank + 0.75
        expected_sums = sorted([3.0 * r + 0.75 for r in range(3) if r != rank])
        got = w_vec.results[rank]
        for g, e in zip(got, expected_sums):
            assert abs(g - e) < 1e-5, f'vec rank {rank}: got {got}, expected {expected_sums}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_ring_collect(rank: int, world_size: int, dist_module, my_payload: Tensor) -> list:
    left = (rank - 1) % world_size
    right = (rank + 1) % world_size
    buf = my_payload.clone()
    received = []
    for _ in range(world_size - 1):
        dist_module.send(buf, dst=right)
        in_buf = t.zeros_like(buf)
        dist_module.recv(in_buf, src=left)
        received.append(in_buf.clone())
        buf = in_buf
    return received
```

**Why `world_size - 1` rotations.** After 1 rotation, every rank has its left-neighbor's original payload. After 2 rotations, every rank has its left-left-neighbor's payload (the prior payload was passed on). After `W-1` rotations, every OTHER rank's payload has visited every rank exactly once.

**`buf = in_buf` instead of `buf.copy_(in_buf)`.** Either works semantically. The rebind is more idiomatic in Python and lets the garbage collector reclaim the old buffer; the in-place copy is marginally faster but allocates an extra tensor at `zeros_like`.

**Deadlock-safety on real gloo.** Every rank calls send THEN recv in the same order, but gloo's send is non-blocking for small tensors — it copies to an internal buffer and returns. The recv then drains. For large tensors (>~64KB), gloo blocks the send until the matching recv posts — which would deadlock this code. The ring needs `isend`/`irecv` for production scale.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()